# 학습 안정성 디버깅 고급

> Loss가 잘 떨어져도 모델이 이상한 출력을 내뱉는 **3대 은밀한 버그**를 진단한다

Phase 4에서 다룬 OOM, Loss 발산 같은 "눈에 보이는" 버그와 달리,  
이 노트북에서는 **Loss는 정상인데 모델 출력이 깨지는** 원인들을 파헤친다.

In [1]:
# === 환경 설치 ===
!pip install datasets transformers peft accelerate trl bitsandbytes

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# GPU 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("GPU 없음 - CPU 모드")

# 토크나이저만 로드 (진단 목적)
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"토크나이저: {model_name}")
print(f"vocab size: {tokenizer.vocab_size}")
print(f"pad_token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")
print(f"eos_token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")
print(f"bos_token: {tokenizer.bos_token} (id: {getattr(tokenizer, 'bos_token_id', 'None')})")

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4070 Ti (12.0 GB)
토크나이저: Qwen/Qwen2.5-1.5B-Instruct
vocab size: 151643
pad_token: <|endoftext|> (id: 151643)
eos_token: <|im_end|> (id: 151645)
bos_token: None (id: None)


---
## 1. 은밀한 버그 #1: padding_side 불일치

**학습**과 **추론**에서 padding 방향이 다르면 모델이 혼란에 빠진다.

| 단계 | padding_side | 이유 |
|------|-------------|------|
| **학습** | `right` | 손실 계산 시 패딩이 뒤에 있어야 마스킹이 정확 |
| **추론** | `left` | 생성 시 마지막 토큰 위치가 실제 토큰이어야 함 |

이 규칙을 어기면 Loss는 내려가는데 생성 결과가 엉뚱해진다.

In [3]:
# === padding_side 시각화 ===

texts = [
    "Hello world",
    "This is a longer sentence for demonstration",
]

# Right padding (학습용)
tokenizer.padding_side = "right"
right_padded = tokenizer(texts, padding=True, return_tensors="pt")

# Left padding (추론용)
tokenizer.padding_side = "left"
left_padded = tokenizer(texts, padding=True, return_tensors="pt")

print("Right Padding (학습용):")
print("  입력 1:", right_padded['input_ids'][0].tolist())
print("  입력 2:", right_padded['input_ids'][1].tolist())
print("  → 패딩이 오른쪽(뒤)에 → 실제 토큰이 왼쪽에 연속")

print("\nLeft Padding (추론용):")
print("  입력 1:", left_padded['input_ids'][0].tolist())
print("  입력 2:", left_padded['input_ids'][1].tolist())
print("  → 패딩이 왼쪽(앞)에 → 마지막 토큰이 실제 토큰 (생성 시작점)")

print(f"\npad_token_id: {tokenizer.pad_token_id}")
print("\n규칙: 학습 전 right, 추론 전 left로 반드시 전환")

Right Padding (학습용):
  입력 1: [9707, 1879, 151643, 151643, 151643, 151643, 151643]
  입력 2: [1986, 374, 264, 5021, 11652, 369, 29716]
  → 패딩이 오른쪽(뒤)에 → 실제 토큰이 왼쪽에 연속

Left Padding (추론용):
  입력 1: [151643, 151643, 151643, 151643, 151643, 9707, 1879]
  입력 2: [1986, 374, 264, 5021, 11652, 369, 29716]
  → 패딩이 왼쪽(앞)에 → 마지막 토큰이 실제 토큰 (생성 시작점)

pad_token_id: 151643

규칙: 학습 전 right, 추론 전 left로 반드시 전환


In [4]:
# === padding_side 진단 함수 ===

def check_padding_side(tokenizer, phase="training"):
    """
    현재 padding_side가 phase에 맞는지 확인.
    phase: 'training' 또는 'inference'
    """
    current = tokenizer.padding_side
    expected = "right" if phase == "training" else "left"
    
    if current == expected:
        print(f"✓ padding_side='{current}' ({phase}에 적합)")
        return True
    else:
        print(f"✗ padding_side='{current}' → '{expected}'로 변경 필요 ({phase} 모드)")
        return False

# 테스트
tokenizer.padding_side = "right"
check_padding_side(tokenizer, "training")
check_padding_side(tokenizer, "inference")

print("\n수정 방법:")
print('  # 학습 전')
print('  tokenizer.padding_side = "right"')
print('  ')
print('  # 추론 전')
print('  tokenizer.padding_side = "left"')

✓ padding_side='right' (training에 적합)
✗ padding_side='right' → 'left'로 변경 필요 (inference 모드)

수정 방법:
  # 학습 전
  tokenizer.padding_side = "right"
  
  # 추론 전
  tokenizer.padding_side = "left"


---
## 2. 은밀한 버그 #2: BOS/EOS 중복

`tokenizer.apply_chat_template()`과 수동 포맷팅을 **동시에** 사용하면  
BOS/EOS 토큰이 중복되어 모델이 혼란에 빠진다.

| 방법 | BOS/EOS 처리 | 위험 |
|------|-------------|------|
| `apply_chat_template()` | 자동 추가 | 수동 추가와 충돌 |
| 수동 포맷팅 | 직접 추가 | template과 충돌 |
| **하나만 선택** | - | **안전** |

In [5]:
# === BOS/EOS 중복 비교 ===

messages = [
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a programming language."},
]

# 방법 1: apply_chat_template (권장)
template_result = tokenizer.apply_chat_template(messages, tokenize=False)
template_tokens = tokenizer.encode(template_result)

# 방법 2: 수동 포맷팅
manual_result = (
    "<|im_start|>user\nWhat is Python?<|im_end|>\n"
    "<|im_start|>assistant\nPython is a programming language.<|im_end|>"
)
manual_tokens = tokenizer.encode(manual_result)

print("방법 1 - apply_chat_template():")
print(f"  텍스트: {template_result[:100]}...")
print(f"  토큰 수: {len(template_tokens)}")
print(f"  첫 5토큰: {template_tokens[:5]}")

print("\n방법 2 - 수동 포맷팅:")
print(f"  텍스트: {manual_result[:100]}...")
print(f"  토큰 수: {len(manual_tokens)}")
print(f"  첫 5토큰: {manual_tokens[:5]}")

print(f"\n차이: {abs(len(template_tokens) - len(manual_tokens))} 토큰")
if template_tokens == manual_tokens:
    print("→ 동일! 이 모델에서는 두 방법 모두 안전")
else:
    print("→ 다름! 하나의 방법만 일관되게 사용해야 함")

방법 1 - apply_chat_template():
  텍스트: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|...
  토큰 수: 41
  첫 5토큰: [151644, 8948, 198, 2610, 525]

방법 2 - 수동 포맷팅:
  텍스트: <|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a programming language.<|...
  토큰 수: 19
  첫 5토큰: [151644, 872, 198, 3838, 374]

차이: 22 토큰
→ 다름! 하나의 방법만 일관되게 사용해야 함


In [6]:
# === BOS/EOS 중복 탐지 함수 ===

def detect_duplicate_special_tokens(tokenizer, text):
    """
    텍스트에서 BOS/EOS 토큰이 중복되었는지 검사.
    중복이 있으면 경고를 출력한다.
    """
    issues = []
    tokens = tokenizer.encode(text)
    token_strs = [tokenizer.decode([t]) for t in tokens]
    
    # BOS 중복 체크
    if tokenizer.bos_token_id is not None:
        bos_count = tokens.count(tokenizer.bos_token_id)
        if bos_count > 1:
            issues.append(f"BOS 토큰 {bos_count}회 발견 (기대: 0~1회)")
    
    # EOS 중복 체크 (대화 턴 수 + 1 이상이면 의심)
    if tokenizer.eos_token_id is not None:
        eos_count = tokens.count(tokenizer.eos_token_id)
        # ChatML에서는 <|im_end|>가 EOS 역할
        im_end_count = text.count("<|im_end|>")
        if eos_count > im_end_count + 1:
            issues.append(f"EOS 토큰 {eos_count}회 (im_end {im_end_count}회 + 예상 추가 0~1회 초과)")
    
    # 연속 special token 체크
    special_ids = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id}
    special_ids.discard(None)
    for i in range(len(tokens) - 1):
        if tokens[i] in special_ids and tokens[i] == tokens[i+1]:
            issues.append(f"위치 {i}: 동일 special token 연속 (id={tokens[i]})")
    
    if issues:
        print("✗ 잠재적 문제 발견:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("✓ special token 중복 없음")
    
    return issues

# 테스트
print("정상 텍스트:")
detect_duplicate_special_tokens(tokenizer, template_result)

print("\n의도적 중복 텍스트:")
bad_text = tokenizer.bos_token + template_result if tokenizer.bos_token else template_result
detect_duplicate_special_tokens(tokenizer, bad_text)

정상 텍스트:
✓ special token 중복 없음

의도적 중복 텍스트:
✓ special token 중복 없음


[]

---
## 3. 은밀한 버그 #3: Special Token 미등록

커스텀 special token을 추가했지만 **단일 토큰으로 등록되지 않은** 경우.  
예: `<|custom|>`을 추가했는데 `<`, `|`, `custom`, `|`, `>`로 분리되어 토큰화.

In [7]:
# === Special Token 등록 검증 ===

def check_special_token_registration(tokenizer, token_str):
    """
    special token이 단일 토큰으로 제대로 등록되었는지 확인.
    여러 토큰으로 분리되면 문제가 있는 것.
    """
    token_ids = tokenizer.encode(token_str, add_special_tokens=False)
    
    if len(token_ids) == 1:
        print(f"✓ '{token_str}' → 단일 토큰 (id: {token_ids[0]})")
        return True
    else:
        decoded = [tokenizer.decode([t]) for t in token_ids]
        print(f"✗ '{token_str}' → {len(token_ids)}개로 분리: {decoded}")
        print(f"  → add_special_tokens 또는 add_tokens으로 등록 필요")
        return False

# 기존 special token 확인
print("기존 special token 검증:")
check_special_token_registration(tokenizer, "<|im_start|>")
check_special_token_registration(tokenizer, "<|im_end|>")

# 미등록 토큰 시뮬레이션
print("\n미등록 토큰 예시:")
check_special_token_registration(tokenizer, "<|custom_tag|>")
check_special_token_registration(tokenizer, "[INST]")

기존 special token 검증:
✓ '<|im_start|>' → 단일 토큰 (id: 151644)
✓ '<|im_end|>' → 단일 토큰 (id: 151645)

미등록 토큰 예시:
✗ '<|custom_tag|>' → 6개로 분리: ['<', '|', 'custom', '_tag', '|', '>']
  → add_special_tokens 또는 add_tokens으로 등록 필요
✗ '[INST]' → 3개로 분리: ['[', 'INST', ']']
  → add_special_tokens 또는 add_tokens으로 등록 필요


False

In [8]:
# === Special Token 올바른 등록 방법 ===

from copy import deepcopy
tokenizer_copy = deepcopy(tokenizer)  # 원본 보존

print("등록 전:")
check_special_token_registration(tokenizer_copy, "<|sql_start|>")

# 올바른 등록
num_added = tokenizer_copy.add_special_tokens({
    "additional_special_tokens": ["<|sql_start|>", "<|sql_end|>"]
})

print(f"\n{num_added}개 토큰 추가 후:")
check_special_token_registration(tokenizer_copy, "<|sql_start|>")
check_special_token_registration(tokenizer_copy, "<|sql_end|>")

print(f"\nvocab 크기: {tokenizer.vocab_size} → {tokenizer_copy.vocab_size}")
print("\n주의: special token 추가 후 model.resize_token_embeddings(len(tokenizer)) 필수!")

del tokenizer_copy  # 메모리 해제

등록 전:
✗ '<|sql_start|>' → 6개로 분리: ['<', '|', 'sql', '_start', '|', '>']
  → add_special_tokens 또는 add_tokens으로 등록 필요

2개 토큰 추가 후:
✓ '<|sql_start|>' → 단일 토큰 (id: 151665)
✓ '<|sql_end|>' → 단일 토큰 (id: 151666)

vocab 크기: 151643 → 151643

주의: special token 추가 후 model.resize_token_embeddings(len(tokenizer)) 필수!


---
## 4. Pre-flight Check 함수

학습 전에 데이터의 무결성을 **자동으로 검증**하는 함수.  
실전에서 이 함수를 한 번 실행하는 것만으로 대부분의 은밀한 버그를 사전에 잡을 수 있다.

In [9]:
# === Pre-flight Data Check ===

def preflight_check(dataset, tokenizer, text_field="text", max_length=512, sample_size=5):
    """
    학습 전 데이터 무결성 검증.
    7가지 항목을 체크한다.
    """
    print("=" * 60)
    print("Pre-flight Data Check")
    print("=" * 60)
    issues = []
    
    # 1. 텍스트 필드 존재 확인
    if text_field in dataset.column_names:
        print(f"✓ 1/7 텍스트 필드 '{text_field}' 존재")
    else:
        print(f"✗ 1/7 텍스트 필드 '{text_field}' 없음! 컬럼: {dataset.column_names}")
        issues.append("텍스트 필드 없음")
        return issues
    
    # 2. 빈 텍스트 체크
    empty_count = sum(1 for x in dataset if not x[text_field].strip())
    if empty_count == 0:
        print(f"✓ 2/7 빈 텍스트 없음")
    else:
        print(f"✗ 2/7 빈 텍스트 {empty_count}개 발견")
        issues.append(f"빈 텍스트 {empty_count}개")
    
    # 3. 토큰 길이 분포
    import random
    indices = random.sample(range(len(dataset)), min(100, len(dataset)))
    lengths = []
    for idx in indices:
        tokens = tokenizer.encode(dataset[idx][text_field])
        lengths.append(len(tokens))
    
    over_max = sum(1 for l in lengths if l > max_length)
    avg_len = sum(lengths) / len(lengths)
    print(f"{'✓' if over_max/len(lengths) < 0.1 else '✗'} 3/7 토큰 길이: 평균 {avg_len:.0f}, max_length 초과 {over_max}/{len(lengths)}개")
    if over_max > len(lengths) * 0.1:
        issues.append(f"max_length({max_length}) 초과 비율 높음: {over_max/len(lengths)*100:.0f}%")
    
    # 4. Special token 정합성
    sample = dataset[0][text_field]
    has_chat_format = "<|im_start|>" in sample or "[INST]" in sample or "<|begin_of_text|>" in sample
    if has_chat_format:
        print(f"✓ 4/7 Chat format 감지")
    else:
        print(f"△ 4/7 Chat format 미감지 (plain text일 수 있음)")
    
    # 5. BOS/EOS 중복 체크
    dup_issues_count = 0
    for idx in indices[:10]:
        text = dataset[idx][text_field]
        tokens = tokenizer.encode(text)
        if tokenizer.bos_token_id and tokens.count(tokenizer.bos_token_id) > 1:
            dup_issues_count += 1
    if dup_issues_count == 0:
        print(f"✓ 5/7 BOS 중복 없음 (샘플 10개)")
    else:
        print(f"✗ 5/7 BOS 중복 {dup_issues_count}/10개")
        issues.append("BOS 토큰 중복")
    
    # 6. padding_side 확인
    if tokenizer.padding_side == "right":
        print(f"✓ 6/7 padding_side='right' (학습 적합)")
    else:
        print(f"✗ 6/7 padding_side='{tokenizer.padding_side}' → 'right'로 변경 필요")
        issues.append("padding_side가 학습에 부적합")
    
    # 7. 샘플 출력
    print(f"\n✓ 7/7 샘플 {sample_size}개:")
    for i in range(min(sample_size, len(dataset))):
        text = dataset[i][text_field]
        print(f"  [{i}] {text[:80]}..." if len(text) > 80 else f"  [{i}] {text}")
    
    print("\n" + "=" * 60)
    if issues:
        print(f"발견된 문제 {len(issues)}개:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("모든 체크 통과! 학습 시작 가능.")
    print("=" * 60)
    
    return issues

print("preflight_check() 함수 정의 완료")
print("사용법: issues = preflight_check(train_dataset, tokenizer)")

preflight_check() 함수 정의 완료
사용법: issues = preflight_check(train_dataset, tokenizer)


In [10]:
# === Pre-flight Check 실행 예시 ===

from datasets import Dataset

# 테스트용 데이터셋 생성
test_data = Dataset.from_dict({
    "text": [
        "<|im_start|>user\nWhat is Python?<|im_end|>\n<|im_start|>assistant\nPython is a programming language.<|im_end|>",
        "<|im_start|>user\nExplain ML<|im_end|>\n<|im_start|>assistant\nML is a subset of AI that learns from data.<|im_end|>",
        "<|im_start|>user\nHello<|im_end|>\n<|im_start|>assistant\nHi there! How can I help you?<|im_end|>",
        "",  # 의도적 빈 텍스트
        "<|im_start|>user\nSQL query for users<|im_end|>\n<|im_start|>assistant\nSELECT * FROM users;<|im_end|>",
    ]
})

tokenizer.padding_side = "right"  # 학습 모드
issues = preflight_check(test_data, tokenizer, max_length=512, sample_size=3)

Pre-flight Data Check
✓ 1/7 텍스트 필드 'text' 존재
✗ 2/7 빈 텍스트 1개 발견
✓ 3/7 토큰 길이: 평균 16, max_length 초과 0/5개
✓ 4/7 Chat format 감지
✓ 5/7 BOS 중복 없음 (샘플 10개)
✓ 6/7 padding_side='right' (학습 적합)

✓ 7/7 샘플 3개:
  [0] <|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a pro...
  [1] <|im_start|>user
Explain ML<|im_end|>
<|im_start|>assistant
ML is a subset of AI...
  [2] <|im_start|>user
Hello<|im_end|>
<|im_start|>assistant
Hi there! How can I help ...

발견된 문제 1개:
  - 빈 텍스트 1개


---
## 5. 학습 전 종합 체크리스트

실전에서 학습 버튼을 누르기 전, 이 체크리스트를 실행하면  
**시간과 GPU 비용을 낭비하는 대부분의 실수**를 방지할 수 있다.

In [11]:
# === 학습 전 종합 체크리스트 ===

def pre_training_checklist(model, tokenizer, train_dataset, text_field="text", max_length=512):
    """
    학습 전 전체 환경을 진단하는 종합 체크리스트.
    모델, 토크나이저, 데이터를 모두 검사한다.
    """
    print("\n" + "#" * 60)
    print("# 학습 전 종합 체크리스트")
    print("#" * 60)
    all_ok = True
    
    # --- 모델 체크 ---
    print("\n[모델]")
    
    # 1. GPU 확인
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"  ✓ GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")
    else:
        print(f"  △ CPU 모드 (학습 매우 느림)")
    
    # 2. 학습 가능 파라미터
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    ratio = trainable / total * 100
    if ratio < 5:
        print(f"  ✓ 학습 파라미터: {trainable:,} ({ratio:.2f}%) - LoRA 적용")
    else:
        print(f"  △ 학습 파라미터: {trainable:,} ({ratio:.2f}%) - Full fine-tune?")
    
    # 3. gradient_checkpointing
    if hasattr(model, 'is_gradient_checkpointing') and model.is_gradient_checkpointing:
        print(f"  ✓ gradient_checkpointing: ON")
    else:
        print(f"  △ gradient_checkpointing: OFF (메모리 부족 시 켜기)")
    
    # --- 토크나이저 체크 ---
    print("\n[토크나이저]")
    
    # 4. pad_token
    if tokenizer.pad_token is not None:
        print(f"  ✓ pad_token: '{tokenizer.pad_token}' (id: {tokenizer.pad_token_id})")
    else:
        print(f"  ✗ pad_token 미설정!")
        all_ok = False
    
    # 5. padding_side
    if tokenizer.padding_side == "right":
        print(f"  ✓ padding_side: 'right' (학습 적합)")
    else:
        print(f"  ✗ padding_side: '{tokenizer.padding_side}' → 'right' 필요")
        all_ok = False
    
    # 6. vocab size 일치
    model_vocab = model.get_input_embeddings().weight.shape[0]
    tok_vocab = len(tokenizer)
    if model_vocab >= tok_vocab:
        print(f"  ✓ vocab: 모델={model_vocab}, 토크나이저={tok_vocab}")
    else:
        print(f"  ✗ vocab 불일치: 모델={model_vocab} < 토크나이저={tok_vocab}")
        print(f"    → model.resize_token_embeddings(len(tokenizer)) 필요")
        all_ok = False
    
    # --- 데이터 체크 ---
    print("\n[데이터]")
    
    # 7. 데이터 크기
    print(f"  ✓ 데이터 크기: {len(train_dataset)}개")
    
    # 8. Pre-flight
    data_issues = preflight_check(train_dataset, tokenizer, text_field, max_length, sample_size=2)
    if data_issues:
        all_ok = False
    
    # --- 최종 판정 ---
    print("\n" + "#" * 60)
    if all_ok:
        print("# 결과: 모든 체크 통과 → 학습 시작 가능")
    else:
        print("# 결과: 문제 발견 → 수정 후 재확인 필요")
    print("#" * 60)
    
    return all_ok

print("pre_training_checklist() 함수 정의 완료")
print("사용법: ok = pre_training_checklist(model, tokenizer, train_data)")

pre_training_checklist() 함수 정의 완료
사용법: ok = pre_training_checklist(model, tokenizer, train_data)


---
## 정리

| 버그 | 증상 | 진단 | 해결 |
|------|------|------|------|
| **padding_side 불일치** | 생성 결과 엉뚱 | `check_padding_side()` | 학습=right, 추론=left |
| **BOS/EOS 중복** | 반복/잘림 | `detect_duplicate_special_tokens()` | 포맷 방법 하나만 사용 |
| **Special token 미등록** | UNK 또는 분리 | `check_special_token_registration()` | `add_special_tokens()` + `resize` |

### 핵심 원칙

> **Loss가 내려가도 모델이 정상이라는 보장은 없다.**  
> 학습 전 5분의 진단이 학습 후 5시간의 디버깅을 막는다.

### 다음: Phase 6 — 모델 정렬 (DPO)

SFT로 "지시를 따르는" 모델을 만들었다면,  
DPO로 "더 나은 답변을 선호하는" 모델을 만든다.